In [1]:
from pathlib import Path
import random
import shutil
from collections import Counter

# ============================================================
# Configuration
# ============================================================

NUM_VAL = 250
NUM_CONFIRM = 1000

TIMIT_ROOT = Path("/shared/data_zfs/blue2959/TIMIT")
TRAIN_ROOT = TIMIT_ROOT / "TRAIN"

VAL_ROOT = TIMIT_ROOT.parent / f"TIMIT_val_{NUM_VAL}"
CONFIRM_ROOT = TIMIT_ROOT.parent / f"TIMIT_confirm_{NUM_CONFIRM}"

VAL_TRAIN_ROOT = VAL_ROOT / "TRAIN"
CONFIRM_TRAIN_ROOT = CONFIRM_ROOT / "TRAIN"

RANDOM_SEED = 42

print(f"Source        : {TRAIN_ROOT}")
print(f"Validation    : {VAL_TRAIN_ROOT}")
print(f"Confirm set   : {CONFIRM_TRAIN_ROOT}")

Source        : /shared/data_zfs/blue2959/TIMIT/TRAIN
Validation    : /shared/data_zfs/blue2959/TIMIT_val_250/TRAIN
Confirm set   : /shared/data_zfs/blue2959/TIMIT_confirm_1000/TRAIN


In [2]:
all_wavs = sorted(TRAIN_ROOT.glob("DR*/*/*.WAV"))

if not all_wavs:
    raise RuntimeError(f"No WAV files found under: {TRAIN_ROOT}")

if len(all_wavs) < NUM_VAL + NUM_CONFIRM:
    raise RuntimeError(
        f"Not enough utterances: found={len(all_wavs)}, "
        + f"required={NUM_VAL + NUM_CONFIRM}"
    )

print(f"Total TRAIN utterances: {len(all_wavs)}")
print("Examples:")

for wav_path in all_wavs[:5]:
    print(" ", wav_path.relative_to(TRAIN_ROOT))

Total TRAIN utterances: 4620
Examples:
  DR1/FCJF0/SA1.WAV
  DR1/FCJF0/SA2.WAV
  DR1/FCJF0/SI1027.WAV
  DR1/FCJF0/SI1657.WAV
  DR1/FCJF0/SI648.WAV


In [3]:
rng = random.Random(RANDOM_SEED)

shuffled_wavs = all_wavs.copy()
rng.shuffle(shuffled_wavs)

val_wavs = sorted(shuffled_wavs[:NUM_VAL])
confirm_wavs = sorted(
    shuffled_wavs[NUM_VAL : NUM_VAL + NUM_CONFIRM]
)

val_relative_paths = {
    path.relative_to(TRAIN_ROOT)
    for path in val_wavs
}
confirm_relative_paths = {
    path.relative_to(TRAIN_ROOT)
    for path in confirm_wavs
}

assert len(val_wavs) == NUM_VAL
assert len(confirm_wavs) == NUM_CONFIRM
assert val_relative_paths.isdisjoint(confirm_relative_paths)

print(f"Validation utterances : {len(val_wavs)}")
print(f"Confirm utterances    : {len(confirm_wavs)}")
print(
    "Overlap              :",
    len(val_relative_paths & confirm_relative_paths),
)

Validation utterances : 250
Confirm utterances    : 1000
Overlap              : 0


In [4]:
def count_dialect_regions(wav_paths):
    return Counter(
        path.relative_to(TRAIN_ROOT).parts[0]
        for path in wav_paths
    )


print("Validation dialect distribution")
print(dict(sorted(count_dialect_regions(val_wavs).items())))

print("\nConfirm-set dialect distribution")
print(dict(sorted(count_dialect_regions(confirm_wavs).items())))

Validation dialect distribution
{'DR1': 25, 'DR2': 49, 'DR3': 38, 'DR4': 33, 'DR5': 32, 'DR6': 21, 'DR7': 40, 'DR8': 12}

Confirm-set dialect distribution
{'DR1': 85, 'DR2': 152, 'DR3': 161, 'DR4': 150, 'DR5': 152, 'DR6': 76, 'DR7': 176, 'DR8': 48}


In [5]:
def copy_selected_utterances(
    selected_wavs: list[Path],
    destination_train_root: Path,
) -> None:
    """
    선택된 utterance의 모든 companion 파일을 복사합니다.

    예:
        TRAIN/DR1/FCJF0/SA1.WAV
        TRAIN/DR1/FCJF0/SA1.PHN
        TRAIN/DR1/FCJF0/SA1.TXT
        TRAIN/DR1/FCJF0/SA1.WRD
    """

    if destination_train_root.exists():
        raise FileExistsError(
            f"Destination already exists: {destination_train_root}\n"
            + "기존 결과를 덮어쓰지 않도록 실행을 중단합니다."
        )

    for wav_path in selected_wavs:
        relative_wav_path = wav_path.relative_to(TRAIN_ROOT)
        relative_parent = relative_wav_path.parent

        destination_dir = (
            destination_train_root / relative_parent
        )
        destination_dir.mkdir(parents=True, exist_ok=True)

        # 동일 발화 stem을 가진 WAV/PHN/TXT/WRD 등을 전부 복사
        companion_files = sorted(
            wav_path.parent.glob(f"{wav_path.stem}.*")
        )

        if not companion_files:
            raise RuntimeError(
                f"No files found for utterance: {wav_path}"
            )

        for source_path in companion_files:
            destination_path = destination_dir / source_path.name
            shutil.copy2(source_path, destination_path)


copy_selected_utterances(
    selected_wavs=val_wavs,
    destination_train_root=VAL_TRAIN_ROOT,
)

copy_selected_utterances(
    selected_wavs=confirm_wavs,
    destination_train_root=CONFIRM_TRAIN_ROOT,
)

print("Copy completed.")

Copy completed.


In [6]:
copied_val_wavs = sorted(
    VAL_TRAIN_ROOT.glob("DR*/*/*.WAV")
)
copied_confirm_wavs = sorted(
    CONFIRM_TRAIN_ROOT.glob("DR*/*/*.WAV")
)

copied_val_relative = {
    path.relative_to(VAL_TRAIN_ROOT)
    for path in copied_val_wavs
}
copied_confirm_relative = {
    path.relative_to(CONFIRM_TRAIN_ROOT)
    for path in copied_confirm_wavs
}

assert len(copied_val_wavs) == NUM_VAL
assert len(copied_confirm_wavs) == NUM_CONFIRM

assert copied_val_relative == val_relative_paths
assert copied_confirm_relative == confirm_relative_paths
assert copied_val_relative.isdisjoint(copied_confirm_relative)

print("Validation set verified :", len(copied_val_wavs))
print("Confirm set verified    :", len(copied_confirm_wavs))
print("Overlap                 :", 0)
print()
print(f"Validation output: {VAL_ROOT}")
print(f"Confirm output   : {CONFIRM_ROOT}")

Validation set verified : 250
Confirm set verified    : 1000
Overlap                 : 0

Validation output: /shared/data_zfs/blue2959/TIMIT_val_250
Confirm output   : /shared/data_zfs/blue2959/TIMIT_confirm_1000


In [7]:
def save_split_list(
    wav_paths: list[Path],
    output_path: Path,
) -> None:
    relative_paths = sorted(
        path.relative_to(TRAIN_ROOT).as_posix()
        for path in wav_paths
    )

    output_path.write_text(
        "\n".join(relative_paths) + "\n",
        encoding="utf-8",
    )


save_split_list(
    val_wavs,
    VAL_ROOT / "split_list.txt",
)

save_split_list(
    confirm_wavs,
    CONFIRM_ROOT / "split_list.txt",
)

print(VAL_ROOT / "split_list.txt")
print(CONFIRM_ROOT / "split_list.txt")

/shared/data_zfs/blue2959/TIMIT_val_250/split_list.txt
/shared/data_zfs/blue2959/TIMIT_confirm_1000/split_list.txt
